# Paragraph Chunking on WHO Medical Guidelines

## Objective
In this notebook we practically evaluate **paragraph chunking** on 10 WHO medical guideline PDFs.
 **No cleaning is applied** to the extracted text, so we can see how paragraph chunking performs on raw PDF output.

## Plan
1. Load the PDFs
2. Build paragraph chunks (raw baseline, then token-limited variants)
3. Add token limits: split long paragraphs, merge short ones (5 variants in total)
4. Build the evaluation set (40 questions with verified answer phrases)
5. Retrieval: BM25, dense (MiniLM) and hybrid (BM25 + dense with RRF)
6. Evaluation: Precision@5, Recall@5, Hit@5 and MRR for every variant and retriever
7. Inspect results and final verdict

## Step 1: Data Loading
We extract the **raw text** of every PDF with PyMuPDF, without any cleaning.
The text is stored page by page so each chunk can later carry its document name and page number.

In [1]:

!pip install pymupdf rank_bm25 sentence-transformers nltk


Defaulting to user installation because normal site-packages is not writeable


In [2]:
import pymupdf
import os
import pandas as pd

PDF_DIR = "data"

docs = []  # one record per page: {"doc_name", "page", "text"}

for fname in sorted(os.listdir(PDF_DIR)):
    if not fname.lower().endswith(".pdf"):
        continue
    pdf = pymupdf.open(os.path.join(PDF_DIR, fname))
    for page_num, page in enumerate(pdf, start=1):
        docs.append({
            "doc_name": fname.strip(),
            "page": page_num,
            "text": page.get_text("text", sort=True),  # raw text, no cleaning
        })
    print(f"{fname.strip()}: {len(pdf)} pages")

print("\nTotal pages:", len(docs))

Cholera .pdf: 124 pages
Covid_19.pdf: 149 pages
HIV.pdf: 50 pages
Mental_health.pdf: 184 pages
cancer.pdf: 50 pages
diabetes.pdf: 12 pages
hepatitis.pdf: 94 pages
hypertension.pdf: 61 pages
malaria.pdf: 494 pages
tuberculosis.pdf: 90 pages

Total pages: 1308


Every PDF is read page by page with `page.get_text("text", sort=True)`.
`sort=True` orders the text blocks top-to-bottom and left-to-right, and it also inserts blank lines
(`\n\n`) between layout blocks. No cleaning is applied.

### Why `sort=True`?
With the default `page.get_text()`, none of the 10 PDFs contained a single `\n\n`, so paragraph
splitting was impossible (one giant chunk per page). With `sort=True`, paragraph-like units appear.
These boundaries come from the PDF layout, so a unit can be a paragraph, a heading, a bullet or a fragment.

### Corpus notes
- `malaria.pdf` has 494 pages while `diabetes.pdf` has only 12, so the corpus is imbalanced.
- `diabetes.pdf` has a broken font encoding (for example "Treatment options" is extracted as
  `7UHDWP HQW\x1f RSWLRQV`). It is kept as-is because no cleaning is allowed.

### Saving the raw text
The extracted text of each document is saved to `raw_text/` so it can be inspected manually.
Each page starts with a `[PAGE n]` marker.

In [3]:
OUT_DIR = "raw_text"
os.makedirs(OUT_DIR, exist_ok=True)

for doc_name in sorted({d["doc_name"] for d in docs}):
    stem = os.path.splitext(doc_name)[0].replace(" ", "_")
    with open(os.path.join(OUT_DIR, f"{stem}.txt"), "w", encoding="utf-8") as f:
        for d in docs:
            if d["doc_name"] == doc_name:
                f.write(f"\n[PAGE {d['page']}]\n")
                f.write(d["text"])

print(sorted(os.listdir(OUT_DIR)))

['Cholera_.txt', 'Covid_19.txt', 'HIV.txt', 'Mental_health.txt', 'cancer.txt', 'diabetes.txt', 'hepatitis.txt', 'hypertension.txt', 'malaria.txt', 'tuberculosis.txt']


## Step 2: Baseline paragraph chunks (Variant raw)

### Method
- Each page's raw text is split on blank lines (`\n\n`); every non-empty piece becomes one chunk.
- No minimum or maximum size is applied in this variant. It is the pure paragraph baseline.
- Each chunk keeps metadata: chunk id, document name and page number.
- Token lengths are measured with the tokenizer of the embedding model, because the model
  silently ignores everything beyond its maximum sequence length

### Why measure token lengths first?
By looking at the length distribution we can see how many chunks are too short to be useful
(page numbers, headings) and how many are too long for the embedding model.
These numbers justify the parameter choices in the next variants.

In [4]:
chunks_a = []
chunk_id = 0

for d in docs:
    for para in d["text"].split("\n\n"):
        para = para.strip()
        if not para:
            continue
        chunks_a.append({
            "chunk_id": chunk_id,
            "doc_name": d["doc_name"],
            "page": d["page"],
            "text": para,
        })
        chunk_id += 1

print("Total chunks:", len(chunks_a))

Total chunks: 13731


#### We load the tokenizer of `bge-small-en-v1.5` and count the tokens of every raw chunk. All later length limits (256, 512, 30) are expressed in these tokens. MiniLM and BGE use very similar BERT-style tokenizers, so one tokenizer is used for counting.

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")

def count_tokens(texts):
    enc = tokenizer(texts, add_special_tokens=False)["input_ids"]
    return [len(x) for x in enc]

df_a = pd.DataFrame(chunks_a)
df_a["tokens"] = count_tokens(df_a["text"].tolist())

df_a["tokens"].describe()

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (531 > 512). Running this sequence through the model will result in indexing errors


count    13731.000000
mean        65.717938
std        115.771132
min          0.000000
25%          9.000000
50%         31.000000
75%         80.000000
max       1906.000000
Name: tokens, dtype: float64

### Chunk length overview
We count how many chunks are very short or exceed the embedding model limits,
overall and per document.

In [6]:
summary = pd.DataFrame({
    "chunks": [len(df_a)],
    "under_10_tokens": [(df_a["tokens"] < 10).sum()],
    "under_30_tokens": [(df_a["tokens"] < 30).sum()],
    "over_256_tokens": [(df_a["tokens"] > 256).sum()],
    "over_512_tokens": [(df_a["tokens"] > 512).sum()],
})
display(summary)

per_doc = df_a.groupby("doc_name")["tokens"].agg(
    chunks="count", mean="mean", median="median", max="max"
).round(1)
display(per_doc)

,chunks,under_10_tokens,under_30_tokens,over_256_tokens,over_512_tokens
0,13731,3549,6667,531,168


,chunks,mean,median,max
doc_name,,,,
Cholera .pdf,397,80.6,32.0,495
Covid_19.pdf,2422,53.3,39.0,942
HIV.pdf,451,59.5,17.0,1906
Mental_health.pdf,1363,77.9,19.0,1375
cancer.pdf,362,58.3,28.0,954
diabetes.pdf,162,28.4,21.0,147
hepatitis.pdf,570,101.7,25.0,1576
hypertension.pdf,661,54.9,36.0,594
malaria.pdf,6700,64.2,34.0,1785


#### A look at the extremes of the length distribution (the five shortest chunks and the first 300 characters of the three longest):

In [7]:
print("=== 5 shortest chunks ===")
for t in df_a.nsmallest(5, "tokens")["text"]:
    print(repr(t))

print("\n=== 3 longest chunks (first 300 chars) ===")
for _, r in df_a.nlargest(3, "tokens").iterrows():
    print(r["doc_name"], "| page", r["page"], "|", r["tokens"], "tokens")
    print(repr(r["text"][:300]))
    print()

=== 5 shortest chunks ===
'\ufeff\ufeff\ufeff'
'i'
'CONTENTS'
'v'
'vii'

=== 3 longest chunks (first 300 chars) ===
HIV.pdf | page 36 | 1906 tokens
'19.  Naar-King S, Wright K, Parsons JT, Frey M, Templin T, Lam P et al. Healthy choices: motivational enhancement therapy for\n                health risk behaviors in HIV-positive youth. AIDS Educ Prev. 2006;18(1):1–11 (https://doi.org/10.1521/aeap.2006.18.1.1).\n            20.  Naar-King S, Lam P, '

malaria.pdf | page 417 | 1785 tokens
'158. [182]. Due to the absence of cases in the reference group, it was not possible to calculate the incidence rate ratio in vaccine\n    recipients. There were no cases of cerebral malaria in the SMC alone group, 4 cases in the RTS,S vaccine alone group (0.723 cases\n   per 1000 PYAR; 95%CI 0.271 to '

malaria.pdf | page 415 | 1764 tokens
'(May – August, prior to the malaria season), with a fourth dose at month 12. Group 3 [n=150]: control group received 3 doses Rabivax-\n   S rabies vaccine at months 

### Findings from Raw Variant 
- Median chunk length is 31 tokens; 26% of chunks have fewer than 10 tokens and 49% fewer than 30.
  The shortest chunks are page numbers, invisible characters and headings.
- 531 chunks exceed 256 tokens and 168 exceed 512 tokens. The longest ones are reference lists and
  evidence tables, not real paragraphs. An embedding model would silently ignore everything after its limit.

### Rules
- **Split (max_tokens):** a paragraph longer than `max_tokens` is split at sentence boundaries and the
  sentences are packed together until the limit is reached. If a single "sentence" is still too long
  (for example a reference list), it is cut into windows of `max_tokens` tokens.
- **Merge (min_tokens):** a chunk shorter than `min_tokens` is merged with the next paragraph of the same
  page, as long as the merged chunk does not exceed `max_tokens`.
- Chunks never cross page boundaries, so page metadata stays correct.

### Variants
| Variant | max_tokens | min_tokens |
|---|---|---|
| raw| none | none |
| split256 | 256 | none |
| split512| 512 | none |
| split256+merge30 | 256 | 30 |
| split512+merge30 | 512 | 30 |

## Step 3: Adding token limits (split and merge variants)

In [8]:
import re
from functools import lru_cache

@lru_cache(maxsize=None)
def n_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def split_long(text, max_tokens):
    """Split a paragraph that is longer than max_tokens into pieces <= max_tokens."""
    if n_tokens(text) <= max_tokens:
        return [text]

    sentences = re.split(r"(?<=[.!?])\s+", text)
    pieces, current = [], ""
    for sent in sentences:
        if n_tokens(sent) > max_tokens:
            # a single sentence is too long: flush, then cut it into token windows
            if current:
                pieces.append(current)
                current = ""
            ids = tokenizer(sent, add_special_tokens=False)["input_ids"]
            for i in range(0, len(ids), max_tokens):
                pieces.append(tokenizer.decode(ids[i:i + max_tokens]))
            continue
        candidate = f"{current} {sent}".strip()
        if current and n_tokens(candidate) > max_tokens:
            pieces.append(current)
            current = sent
        else:
            current = candidate
    if current:
        pieces.append(current)
    return pieces

def build_chunks(docs, max_tokens=None, min_tokens=None):
    chunks, chunk_id = [], 0
    for d in docs:
        # 1) paragraphs of this page
        paras = [p.strip() for p in d["text"].split("\n\n") if p.strip()]

        # 2) split the long ones
        if max_tokens:
            split_paras = []
            for p in paras:
                split_paras.extend(split_long(p, max_tokens))
            paras = split_paras

        # 3) merge the short ones (within the same page)
        if min_tokens:
            merged, buffer = [], ""
            for p in paras:
                if not buffer:
                    buffer = p
                elif n_tokens(buffer) < min_tokens and (
                    not max_tokens or n_tokens(buffer + "\n\n" + p) <= max_tokens
                ):
                    buffer = buffer + "\n\n" + p
                else:
                    merged.append(buffer)
                    buffer = p
            if buffer:
                merged.append(buffer)
            paras = merged

        for p in paras:
            chunks.append({
                "chunk_id": chunk_id,
                "doc_name": d["doc_name"],
                "page": d["page"],
                "text": p,
            })
            chunk_id += 1
    return chunks
    

In [9]:
variants = {
    "raw":              dict(max_tokens=None, min_tokens=None),
    "split256":         dict(max_tokens=256,  min_tokens=None),
    "split512":         dict(max_tokens=512,  min_tokens=None),
    "split256+merge30": dict(max_tokens=256,  min_tokens=30),
    "split512+merge30": dict(max_tokens=512,  min_tokens=30),
}

all_chunks = {}
rows = []
for name, params in variants.items():
    chunks = build_chunks(docs, **params)
    all_chunks[name] = chunks
    toks = [n_tokens(c["text"]) for c in chunks]
    s = pd.Series(toks)
    rows.append({
        "variant": name,
        **params,
        "chunks": len(chunks),
        "median_tokens": s.median(),
        "mean_tokens": round(s.mean(), 1),
        "under_30": (s < 30).sum(),
        "over_256": (s > 256).sum(),
        "over_512": (s > 512).sum(),
    })

pd.DataFrame(rows)

,variant,max_tokens,min_tokens,chunks,median_tokens,mean_tokens,under_30,over_256,over_512
0,raw,NaN,NaN,13731,31.0,65.7,6667,531,168
1,split256,256.0,NaN,14630,36.0,61.7,6733,2,0
2,split512,512.0,NaN,13953,33.0,64.7,6680,633,0
3,split256+merge30,256.0,30.0,10009,71.0,90.2,1248,2,0
4,split512+merge30,512.0,30.0,9293,68.0,97.1,1164,647,0


### Observations
- Splitting (max_tokens) removes the oversized chunks but does nothing for the short ones.
- Merging (min_tokens) is what fixes the short chunks: the chunk count drops by about 30% and the median length doubles.
- Some short chunks remain (for example the last paragraph of a page), because chunks never cross page boundaries.
- The 2 chunks slightly above 256 tokens come from decode/re-tokenize differences and are negligible.

### Saving the chunks for manual inspection
The chunks of every variant are saved in the `chunks_out/` folder:
- `<variant>.txt`: human-readable, each chunk starts with a header line (chunk id, document, page, token count)
- `<variant>.csv`: one row per chunk, so it can be opened in Excel and filtered or sorted by tokens

In [10]:
import os, re

OUT = "chunks_out"
os.makedirs(OUT, exist_ok=True)

for v, chunks in all_chunks.items():
    safe = re.sub(r"[^\w\-]+", "_", v)   # "split256+merge30" -> "split256_merge30"

    # 1) readable text file
    with open(os.path.join(OUT, f"{safe}.txt"), "w", encoding="utf-8") as f:
        for c in chunks:
            f.write(f"##### CHUNK {c['chunk_id']} | {c['doc_name']} | page {c['page']} | {n_tokens(c['text'])} tokens\n")
            f.write(c["text"] + "\n\n")

    # 2) CSV file
    df = pd.DataFrame(chunks)
    df["tokens"] = [n_tokens(t) for t in df["text"]]
    df.to_csv(os.path.join(OUT, f"{safe}.csv"), index=False, encoding="utf-8-sig")

print(sorted(os.listdir(OUT)))

['overall_results.csv', 'raw.csv', 'raw.txt', 'split256.csv', 'split256.txt', 'split256_merge30.csv', 'split256_merge30.txt', 'split512.csv', 'split512.txt', 'split512_merge30.csv', 'split512_merge30.txt']


## Step 4: Evaluation set

### Design
- 40 questions (4 per document), drafted with an LLM from the extracted text of the 10 documents. Each question is linked to one source document and has one **answer phrase**, plus up to two alternative phrases, copied verbatim from the extracted text.
- A retrieved chunk counts as **relevant** if it comes from the correct document and contains at least one of the answer phrases (case-insensitive, whitespace-normalized).
- This definition does not depend on chunk ids, so all chunking variants are evaluated on the same questions.
- Several chunks can be relevant when the same fact is repeated in the document (for example in a summary and in the full recommendation). The number of relevant chunks per variant is computed later (`rel_count`).
- Each question also has `must_have` keywords, which are used only to audit the answer phrases, not to decide relevance.

### Validation
Before retrieval we check that:
1. every answer phrase can be found in the text of the correct document, and flag phrases that appear many times;
2. every question keeps at least one relevant chunk in every chunking variant, so that no variant loses a question because a phrase is cut across two chunks;
3. the relevant chunks also contain the `must_have` keywords (spot check).

A sample of the questions was also checked manually against the PDFs.

In [11]:
import json

QUESTIONS_PATH = "questions_new.json"   

with open(QUESTIONS_PATH, encoding="utf-8") as f:
    eval_set = json.load(f)

def norm(s):
    return re.sub(r"\s+", " ", s).strip().lower()

def norm_name(name):
    # ignore case and spaces so "Cholera .pdf" and "cholera.pdf" match
    return name.replace(" ", "").lower()

def is_relevant(chunk, q):
    return (
        norm_name(chunk["doc_name"]) == norm_name(q["doc"])
        and norm(q["answer_phrase"]) in norm(chunk["text"])
    )

# check that every document name in the questions exists in the corpus
corpus_docs = {norm_name(d["doc_name"]) for d in docs}
bad = [q["qid"] for q in eval_set if norm_name(q["doc"]) not in corpus_docs]
print("Questions loaded:", len(eval_set))
print("Questions with unknown document name:", bad)

Questions loaded: 40
Questions with unknown document name: []


## Step 5: Retrieval (BM25, Dense, Hybrid)

Three retrievers are built for every chunking variant:

- **BM25**: lexical ranking (`rank_bm25`, `BM25Okapi`) on lowercased word tokens.
- **Dense**: `all-MiniLM-L6-v2` (max 256 tokens), cosine similarity on normalized embeddings. Embeddings are cached on disk, one file per variant.
- **Hybrid**: Reciprocal Rank Fusion of the BM25 and Dense rankings. Each chunk gets `sum(1 / (60 + rank))` over the two top-50 lists.

All 5 chunking variants are evaluated with all 3 retrievers (15 combinations).

In [12]:
import os
import re
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

MINILM = "sentence-transformers/all-MiniLM-L6-v2"


def tok(text):
    return re.findall(r"\w+", text.lower())


class BM25Retriever:
    def __init__(self, chunks):
        self.chunks = chunks
        self.bm25 = BM25Okapi([tok(c["text"]) for c in chunks])

    def search(self, query, k=10):
        scores = self.bm25.get_scores(tok(query))
        top = np.argsort(-scores)[:k]
        return [(self.chunks[i], float(scores[i])) for i in top]


class DenseRetriever:
    def __init__(self, chunks, model_name, max_len, cache_path):
        self.chunks = chunks
        self.model = SentenceTransformer(model_name)
        self.model.max_seq_length = max_len
        if os.path.exists(cache_path):
            self.emb = np.load(cache_path)
            assert len(self.emb) == len(chunks), f"stale embedding cache: {cache_path}"
        else:
            self.emb = self.model.encode(
                [c["text"] for c in chunks],
                batch_size=64,
                normalize_embeddings=True,
                show_progress_bar=True,
            )
            np.save(cache_path, self.emb)

    def search(self, query, k=10):
        q = self.model.encode([query], normalize_embeddings=True)[0]
        scores = self.emb @ q
        top = np.argsort(-scores)[:k]
        return [(self.chunks[i], float(scores[i])) for i in top]


class HybridRetriever:
    def __init__(self, bm25, dense, rrf_k=60, depth=50):
        self.bm25, self.dense = bm25, dense
        self.rrf_k, self.depth = rrf_k, depth

    def search(self, query, k=10):
        fused = {}
        for retriever in (self.bm25, self.dense):
            for rank, (c, _) in enumerate(retriever.search(query, k=self.depth), start=1):
                entry = fused.setdefault(c["chunk_id"], [c, 0.0])
                entry[1] += 1 / (self.rrf_k + rank)
        top = sorted(fused.values(), key=lambda x: -x[1])[:k]
        return [(c, s) for c, s in top]

In [13]:
retrievers = {}
for v, chunks in all_chunks.items():
    bm = BM25Retriever(chunks)
    dn = DenseRetriever(chunks, MINILM, 256, f"emb_minilm_{v}.npy")
    retrievers[(v, "BM25")] = bm
    retrievers[(v, "Dense")] = dn
    retrievers[(v, "Hybrid")] = HybridRetriever(bm, dn)

print({v: len(c) for v, c in all_chunks.items()})
print("retrievers ready:", len(retrievers))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'raw': 13731, 'split256': 14630, 'split512': 13953, 'split256+merge30': 10009, 'split512+merge30': 9293}
retrievers ready: 15


## Step 6: Evaluation

Every evaluation question is run through every (variant, retriever) combination and the top 10 chunks are scored with the relevance rule from Step 4.

- **Precision@k** = relevant chunks in the top k / k
- **Recall@k** = relevant chunks in the top k / all relevant chunks that exist in the variant
- **Hit@k** = 1 if at least one relevant chunk is in the top k, else 0
- **MRR** (Mean Reciprocal Rank) = average of 1 / rank of the first relevant chunk (0 if none in the top 10)

Recall depends on how many relevant chunks a variant contains, which changes with chunk size, so **MRR and Hit@5** are the fairer metrics across variants. Metrics are averaged over all questions (macro average).

In [14]:
KS = (1, 3, 5)


def eval_one(retriever, q, total_rel, depth=10):
    res = retriever.search(q["question"], k=depth)
    flags = [is_relevant(c, q) for c, _ in res]
    first = next((i + 1 for i, f in enumerate(flags) if f), None)

    out = {"RR": 1 / first if first else 0.0}   # reciprocal rank (0 if not in top 10)
    for k in KS:
        hits = sum(flags[:k])
        out[f"Hit@{k}"] = float(hits > 0)
        out[f"P@{k}"] = hits / k
        out[f"R@{k}"] = hits / total_rel if total_rel else 0.0
    return out


# number of relevant chunks per (variant, question)
rel_count = {
    (v, q["qid"]): sum(is_relevant(c, q) for c in chunks)
    for v, chunks in all_chunks.items()
    for q in eval_set
}

per_query = {}
for (v, m), r in retrievers.items():
    rows = [{"qid": q["qid"], **eval_one(r, q, rel_count[(v, q["qid"])])} for q in eval_set]
    per_query[(v, m)] = pd.DataFrame(rows).set_index("qid")

print("evaluated:", len(per_query), "combinations x", len(eval_set), "questions")

evaluated: 15 combinations x 40 questions


In [15]:
rows = []
for (v, m), df in per_query.items():
    rows.append({
        "variant": v,
        "model": m,
        "chunks": len(all_chunks[v]),
        "P@5": df["P@5"].mean(),
        "R@5": df["R@5"].mean(),
        "Hit@5": df["Hit@5"].mean(),  
        "MRR": df["RR"].mean(),        # MRR@10
    })

summary = pd.DataFrame(rows).round(3)
summary = summary.sort_values(["variant", "model"]).reset_index(drop=True)
display(summary)

# Model ke hisab se side-by-side view
for metric in ["R@5", "P@5", "MRR", "Hit@5"]:
    print(f"\n=== {metric} ===")
    display(summary.pivot(index="variant", columns="model", values=metric)
                   .loc[list(variants)])

# Har metric me sabse achha combo
for metric in ["R@5", "P@5", "MRR"]:
    best = summary.loc[summary[metric].idxmax()]
    print(f"Best {metric}: {best['variant']} + {best['model']} = {best[metric]}")

# Sanity check
zero = [(v, qid) for (v, qid), n in rel_count.items() if n == 0]
print("Questions with 0 relevant chunks:", zero)

summary.to_csv("overall_results.csv", index=False)

,variant,model,chunks,P@5,R@5,Hit@5,MRR
0,raw,BM25,13731,0.155,0.535,0.550,0.452
1,raw,Dense,13731,0.180,0.601,0.650,0.507
2,raw,Hybrid,13731,0.185,0.668,0.725,0.565
3,split256,BM25,14630,0.150,0.497,0.525,0.443
4,split256,Dense,14630,0.170,0.570,0.625,0.501
5,split256,Hybrid,14630,0.195,0.680,0.725,0.556
6,split256+merge30,BM25,10009,0.185,0.627,0.650,0.513
7,split256+merge30,Dense,10009,0.190,0.645,0.725,0.569
8,split256+merge30,Hybrid,10009,0.210,0.728,0.775,0.635
9,split512,BM25,13953,0.160,0.560,0.575,0.452



=== R@5 ===


model,BM25,Dense,Hybrid
variant,,,
raw,0.535,0.601,0.668
split256,0.497,0.570,0.680
split512,0.560,0.601,0.686
split256+merge30,0.627,0.645,0.728
split512+merge30,0.622,0.676,0.728



=== P@5 ===


model,BM25,Dense,Hybrid
variant,,,
raw,0.155,0.18,0.185
split256,0.150,0.17,0.195
split512,0.160,0.18,0.195
split256+merge30,0.185,0.19,0.210
split512+merge30,0.175,0.20,0.205



=== MRR ===


model,BM25,Dense,Hybrid
variant,,,
raw,0.452,0.507,0.565
split256,0.443,0.501,0.556
split512,0.452,0.506,0.562
split256+merge30,0.513,0.569,0.635
split512+merge30,0.513,0.574,0.642



=== Hit@5 ===


model,BM25,Dense,Hybrid
variant,,,
raw,0.550,0.650,0.725
split256,0.525,0.625,0.725
split512,0.575,0.650,0.725
split256+merge30,0.650,0.725,0.775
split512+merge30,0.650,0.750,0.775


Best R@5: split256+merge30 + Hybrid = 0.728
Best P@5: split256+merge30 + Hybrid = 0.21
Best MRR: split512+merge30 + Hybrid = 0.642
Questions with 0 relevant chunks: []


In [16]:
from IPython.display import display, HTML
import html

def compare_variants(query, qid=None, method="Hybrid", k=5, show_chunks=True, preview=300):
    q = next((x for x in eval_set if x["qid"] == qid), None) if qid else None
    all_results, rows = {}, []

    for v in all_chunks:
        results = retrievers[(v, method)].search(query, k=max(k, 10))
        all_results[v] = results
        info = {"variant": v, "chunks": len(all_chunks[v])}
        if q:
            flags = [is_relevant(c, q) for c, _ in results]
            total_rel = sum(is_relevant(c, q) for c in all_chunks[v])
            first = next((i + 1 for i, f in enumerate(flags) if f), None)
            hits = sum(flags[:k])
            info.update({
                "relevant_in_corpus": total_rel,
                f"relevant_in_top{k}": hits,
                f"P@{k}": round(hits / k, 2),
                f"R@{k}": round(hits / total_rel, 2) if total_rel else 0,
                "first_rank": first,
                "RR": round(1 / first, 3) if first else 0.0,
            })
        rows.append(info)

    # header
    head = f"<h3 style='margin-bottom:4px'>Query: {html.escape(query)}</h3>"
    head += f"<div><b>Method:</b> {method}</div>"
    if q:
         phrases = q.get("answer_phrases") or [q["answer_phrase"]]
         head += "<div><b>Answer phrase(s):</b> " + " &nbsp;|&nbsp; ".join(html.escape(p) for p in phrases) + "</div>"
    display(HTML(head))
    display(pd.DataFrame(rows).set_index("variant"))

    if not show_chunks:
        return

    for v, results in all_results.items():
        out = (
            "<div style='margin-top:40px'></div>"
            "<div style='background:#1f3a5f;color:#ffffff;padding:10px 14px;"
            "font-size:18px;font-weight:bold;border-radius:6px'>"
            f"VARIANT {v} &nbsp;|&nbsp; {len(all_chunks[v])} chunks</div>"
        )
        for rank, (c, score) in enumerate(results[:k], start=1):
            relevant = bool(q and is_relevant(c, q))
            bg = "#e3f6e8" if relevant else "#f4f4f4"
            border = "#1e9e4a" if relevant else "#bbbbbb"
            tag = " &nbsp;<b style='color:#1e7a3c'>[RELEVANT]</b>" if relevant else ""
            text = html.escape(c["text"][:preview].replace("\n", " "))
            out += (
                f"<div style='background:{bg};color:#111111;border-left:5px solid {border};"
                "padding:8px 12px;margin:8px 0;border-radius:4px'>"
                f"<b>#{rank}</b>{tag} &nbsp;|&nbsp; score {score:.3f} &nbsp;|&nbsp; "
                f"{html.escape(c['doc_name'])} &nbsp;|&nbsp; page {c['page']} &nbsp;|&nbsp; "
                f"{n_tokens(c['text'])} tokens"
                f"<div style='margin-top:4px'>{text}</div></div>"
            )
        display(HTML(out))


In [17]:
qid = "Q20"
q_text = next(x["question"] for x in eval_set if x["qid"] == qid)
compare_variants(q_text, qid=qid, method="Hybrid")

,chunks,relevant_in_corpus,relevant_in_top5,P@5,R@5,first_rank,RR
variant,,,,,,,
raw,13731,1,1,0.2,1.0,2,0.500
split256,14630,1,1,0.2,1.0,2,0.500
split512,13953,1,1,0.2,1.0,2,0.500
split256+merge30,10009,1,1,0.2,1.0,3,0.333
split512+merge30,9293,1,1,0.2,1.0,3,0.333


In [18]:
print(summary.sort_values(["MRR", "R@5"], ascending=False).head(4)[["variant", "model", "MRR", "Hit@5", "R@5", "P@5"]])

best_key = ("split256+merge30", "Hybrid")      # chosen: ties with split512+merge30, but stays within the 256-token limit of MiniLM
best_ret = retrievers[best_key]

             variant   model    MRR  Hit@5    R@5    P@5
14  split512+merge30  Hybrid  0.642  0.775  0.728  0.205
8   split256+merge30  Hybrid  0.635  0.775  0.728  0.210
13  split512+merge30   Dense  0.574  0.750  0.676  0.200
7   split256+merge30   Dense  0.569  0.725  0.645  0.190


### Best configuration
Among all combinations, those within 0.01 MRR of the best one are treated as ties (with 40 questions such differences are noise). From these, only variants whose chunks fit the 256-token limit of MiniLM are considered, and the one with the highest MRR is selected.

In [19]:
# chunks longer than MiniLM's 256-token limit, per variant
over256 = {v: sum(n_tokens(c["text"]) > 256 for c in ch) for v, ch in all_chunks.items()}
summary["over_256"] = summary["variant"].map(over256)

NOISE = 0.01        # MRR differences below this are treated as ties (40 questions)
MAX_OVER = 10       # allow a handful of chunks above the 256-token limit

top_mrr = summary["MRR"].max()
cand = summary[(summary["MRR"] >= top_mrr - NOISE) & (summary["over_256"] <= MAX_OVER)]
cand = cand.sort_values(["MRR", "R@5"], ascending=False)
display(cand[["variant", "model", "MRR", "Hit@5", "R@5", "P@5", "over_256"]])

best = cand.iloc[0]
best_key = (best["variant"], best["model"])
best_ret = retrievers[best_key]
print(f"Selected config: {best_key[0]} + {best_key[1]} | MRR={best['MRR']}  R@5={best['R@5']}  Hit@5={best['Hit@5']}")

,variant,model,MRR,Hit@5,R@5,P@5,over_256
8,split256+merge30,Hybrid,0.635,0.775,0.728,0.21,2


Selected config: split256+merge30 + Hybrid | MRR=0.635  R@5=0.728  Hit@5=0.775


### Answers of the selected configuration
For every evaluation question we show the top 3 retrieved chunks of the selected configuration. Relevant chunks are marked in green, and each header shows how many relevant chunks exist in the document and how many were retrieved.

In [20]:
def show_answers(retriever, variant, k=3, preview=350, depth=10):
    for q in eval_set:
        res = retriever.search(q["question"], k=depth)
        flags = [is_relevant(c, q) for c, _ in res]
        total = rel_count[(variant, q["qid"])]
        in_k, in_depth = sum(flags[:k]), sum(flags)
        first = next((i + 1 for i, f in enumerate(flags) if f), None)
        status = f"found at rank {first}" if first else "NOT found in top 10"

        out = (f"<div style='margin-top:28px;background:#1f3a5f;color:#fff;padding:8px 12px;"
               f"border-radius:6px'><b>{q['qid']}</b> | {html.escape(q['doc'])} | {status}<br>"
               f"{html.escape(q['question'])}</div>"
               f"<div style='margin:4px 0'><b>Answer phrase:</b> {html.escape(q['answer_phrase'])}<br>"
               f"<b>Relevant chunks in doc:</b> {total} &nbsp;|&nbsp; "
               f"<b>retrieved in top-{k}:</b> {in_k} &nbsp;|&nbsp; "
               f"<b>in top-{depth}:</b> {in_depth}</div>")
        for rank, (c, score) in enumerate(res[:k], start=1):
            rel = flags[rank - 1]
            bg, border = ("#e3f6e8", "#1e9e4a") if rel else ("#f4f4f4", "#bbbbbb")
            tag = " <b style='color:#1e7a3c'>[RELEVANT]</b>" if rel else ""
            text = html.escape(c["text"][:preview].replace("\n", " "))
            out += (f"<div style='background:{bg};color:#111;border-left:5px solid {border};"
                    f"padding:8px 12px;margin:6px 0;border-radius:4px'>"
                    f"<b>#{rank}</b>{tag} | {html.escape(c['doc_name'])} | page {c['page']}"
                    f"<div style='margin-top:4px'>{text}</div></div>")
        display(HTML(out))

show_answers(best_ret, best_key[0], k=3)

## Final Verdict
- **Best overall: Hybrid (BM25 + MiniLM, RRF) with split256+merge30**: MRR 0.635, Hit@5 0.775, R@5 0.728, P@5 0.210, with ~30% fewer chunks than split256.
- `split512+merge30` + Hybrid is practically tied (MRR 0.642), but 647 of its chunks exceed the 256-token limit of MiniLM, so split256+merge30 is preferred.
- **Hybrid is the best retriever for every chunking variant** (MRR 0.556 to 0.642), ahead of Dense (0.501 to 0.574) and BM25 (0.443 to 0.513).
- **Merging short paragraphs (min 30 tokens) helps all three retrievers.** For Hybrid, MRR rises from about 0.56 to 0.64 and Hit@5 from 0.725 to 0.775. Splitting alone changes little compared with raw.
- Caveat: 40 questions (1 question = 0.025 in Hit@5), so differences below about 0.03 to 0.05 are within noise.